In [0]:
# 1. Importat librerias iniciales
from pyspark.sql.functions import *

In [0]:

%sql
-- # 2. lectura incial de silver
select * from workspace.weather_silver.weather limit 1

In [0]:
 
%sql
-- # 3. Borrar la base de datos en este punto porque es el script
DROP DATABASE IF EXISTS workspace.weather_gold CASCADE; 

In [0]:

%sql
-- # 4. Crear la base de datos
CREATE DATABASE IF NOT EXISTS workspace.weather_gold
COMMENT 'Capa Gold modelo estrella'

In [0]:
# 4. Crear la tabla workspace.weather_gold.dim_date
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.weather_gold.dim_date (
  date_id INTEGER,
  date DATE,
  year INTEGER,
  month INTEGER,
  day INTEGER,
  quarter INTEGER,
  month_name STRING
)
""")

In [0]:
import pandas as pd

# 5. Leer la capa silver
df_silver = spark.table("workspace.weather_silver.weather")
df_silver = df_silver.toPandas()


# 6. Crear DIM_DATE
dim_date = (
    df_silver[
        ["date_id", "daily_time"]
    ]
    .rename(columns={
        "daily_time": "date"
    })
    .drop_duplicates()
)

# 7. Actualizar a formato datetime para crear nuevos campos 
dim_date["date"] = pd.to_datetime(
    dim_date["date"],
    errors="coerce"
)

# 8. Crear atributos de fecha para la dimension
dim_date["year"] = dim_date["date"].dt.year
dim_date["month"] = dim_date["date"].dt.month
dim_date["day"] = dim_date["date"].dt.day
# Crear nuevos campos
dim_date["quarter"] = dim_date["date"].dt.quarter # corresponde a trimestre
dim_date["month_name"] = dim_date["date"].dt.month_name()

# 9. Ordenar por fecha
dim_date = dim_date.sort_values("date")

# 10. Ver resultado
print(dim_date.head())
print(dim_date.dtypes)
# 11. para darle fomato date a la fecha
dim_date["date"] = dim_date["date"].dt.date

dim_date_spark = spark.createDataFrame(dim_date)
dim_date_spark.printSchema()

# 12. Guardar la dimension dim_date
dim_date_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.weather_gold.dim_date")

In [0]:
%sql
select * from workspace.weather_gold.dim_date 